# Детекция ботов по событиям пользователей — тестовое задание Авито

**Итог:** `submission.csv`, на платформе Precision@Recall≥0.70 = **0.90251**.

Модель — один `HistGradientBoostingClassifier` на 139 признаках:
- 130 признаков описывают поведение самой куки;
- 9 признаков показывают, насколько просмотренные кукой объявления совпадают с объявлениями ботов из прошлых дней.

Ноутбук заново обучает модель; в проверенной среде полный запуск занял около 76 секунд, на другом компьютере время может отличаться. В последней ячейке проверяется, что получился тот же `submission.csv`, что был загружен на платформу: совпадение по SHA-256 (точное совпадение проверено на Linux, Python 3.12.14).

## Как я решал задачу

### 1. Сначала — ловушки в данных

Перед признаками я проверил, нет ли в данных того, что «подсказывает» ответ нечестно или просто мешает.

- **События за окном наблюдения.** Около 17% событий в train лежат после конца окна куки, у ботов — 42%. В test таких событий нет. После окна же находятся все события `captcha_shown`: капчу видела почти половина ботов и 1% людей. Если такие события не отрезать, валидация будет выглядеть отлично, а на test модель провалится. Поэтому все признаки считаются только по событиям внутри окна `[начало, конец)`.
- **Полные дубли.** Это 1.5% строк, у ботов и у людей их доля одинакова. Похоже на технический повтор логирования, поэтому я их удаляю.
- **Мелкие артефакты.** Платформа записана одиннадцатью способами (`web`, `WEB`, `desktop`, …), я привожу её к трём значениям. Строки в файле перемешаны, поэтому я сортирую события по времени. Порядок событий внутри одной секунды восстановить нельзя, и я его не использую.

### 2. Валидация, похожая на реальную проверку

Test — это следующая неделя после train, поэтому случайное разбиение дало бы слишком оптимистичную оценку. Я проверяю модель на четырёх последовательных двухдневных блоках: 12–13, 14–15, 16–17 и 18–19 апреля. Для каждого блока модель учится только на окнах, закончившихся до его начала. Дополнительно смотрю на семидневный блок 13–19 апреля: он ближе всего к ситуации с test.

Метрика — официальная: максимальная precision при recall ≥ 0.70. Куки с одинаковым score не разделяются.

### 3. Признаки поведения (130)

Каждая кука описывается только своими событиями:
- **ритм** — паузы между событиями и их разброс, очень короткие паузы, сессии, активность ночью;
- **поиск** — глубина страниц выдачи, повторы запросов, листание;
- **разнообразие** — сколько разных объявлений, категорий и городов;
- **состав действий** — доли просмотров, фото, контактов, избранного;
- **техническое** — координаты курсора на web, платформа, семейство User-Agent, возраст куки.

Сырую строку User-Agent в модель я не передаю: модель просто запомнила бы конкретные строки.

На этих признаках модель набрала 0.805 на платформе. LightGBM, CatBoost и ансамбли почти ничего не добавили. Из этого я сделал вывод, что упираюсь не в алгоритм, а в информацию, которой не хватает модели.

### 4. Главная находка — «пул» объявлений ботов

Разбирая ошибки, я заметил, что многие боты смотрят объявления из одного и того же ограниченного набора. Отсюда появились признаки пула: какая доля объявлений куки уже встречалась у ботов за прошлые 7 дней, плюс сглаженная доля ботов среди прошлых зрителей объявления.

Здесь легко допустить утечку, поэтому метки берутся только из прошлого:
- обучающая строка видит только предыдущие дни;
- валидационный блок и test видят одну историю, замороженную на момент их начала. Для test это train за 13–19 апреля.

Каждая test-кука обрабатывается независимо от остальных test-кук.

Эти четыре признака подняли результат с 0.805 до 0.895.

### 5. Итоговое улучшение: очищенный пул и LLR

Потом выяснилось, что боты бывают двух типов:
- примерно 60% смотрят объявления из общего набора;
- остальные смотрят случайные объявления, как люди.

Объявления второй группы «засоряют» пул, и обычные пользователи случайно в него попадают. Это давало заметную часть ложных срабатываний. Поэтому я сделал две вещи:
- **очищенный пул** — беру объявления только тех прошлых ботов, у которых не меньше 30% объявлений встречались и у других ботов;
- **признак правдоподобия (LLR)** — учитывает не только попадания куки в пул, но и промахи, и сравнивает, насколько такая картина типична для бота и для человека:

  `LLR = h·log(p_b / p_h) + (k − h)·log((1 − p_b) / (1 − p_h))`

  Здесь k — число объявлений куки, h — сколько из них в пуле, p_b и p_h — доли попаданий у ботов и у людей, посчитанные по истории.

Проверка с отключением признаков по одному показала, что основной вклад даёт именно LLR. Итог — 0.90251 на платформе.

## Что пробовал и что не сработало

| Версия | Что изменено | Локально: блоки / 7 дней | Платформа |
|---|---|---|---:|
| v4 | 130 признаков + пул за 7 дней | 0.892 / 0.878 | 0.89544 |
| v5 | + LLR по неочищенному пулу | 0.899 / 0.883 | 0.89702 |
| **v6 (итог)** | **+ очищенный пул и LLR** | **0.899 / 0.902** | **0.90251** |
| v7 | очищенный пул по всей истории + среднее 10 моделей | 0.903 / 0.905 | 0.89840 |

Не дали устойчивого улучшения:
- другие модели и ансамбли;
- target encoding строки User-Agent;
- дополнительные статистики пауз;
- наивный Байес по отдельным событиям.

Я также проверил, несёт ли информацию порядок событий внутри куки. Статистика соседних событий такая же, как после случайного перемешивания, поэтому дополнительные признаки порядка событий я не добавлял; в базовых признаках остались только агрегаты переходов между соседними поисками.

## Ограничения

Разница между v5, v6 и v7 меньше погрешности одной отправки, которая составляет около ±0.017. Доказать, что v6 лучше двух других, нельзя: я выбрал его по лучшему результату на платформе.

Локальные блоки использовались для многих решений, поэтому локальные оценки, вероятно, немного оптимистичны.

Главная нерешённая проблема — «человекоподобные» боты со случайными объявлениями, особенно на мобильном Chrome. По доступным полям они почти не отличаются от людей.

## Как запустить

Положите `train.csv`, `test.csv` и `events.csv.gz` в папку `data/` и выполните все ячейки по порядку: *Restart Kernel and Run All*. Команды установки для Windows и Linux — в `README.md`.

Флаги в первой ячейке кода:
- `RUN_VALIDATION = False` пропускает валидацию;
- `REPRODUCE_PREVIOUS = True` дополнительно воспроизводит прежние отправки v4, v5 и v7.

Использованы только открытые библиотеки numpy, pandas, scipy и scikit-learn. Внешних API и языковых моделей в решении нет.

## 0. Настройка

Потоки ограничиваю до импорта численных библиотек, а seed фиксирую. Без этого сумма float в разном порядке могла бы дать немного другие score, и `submission.csv` не воспроизводился бы побайтово. Здесь же флаги запуска.

In [1]:
import os

# Ограничение потоков задаётся до импорта численных библиотек.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path
import hashlib
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from threadpoolctl import threadpool_limits

thread_limits = threadpool_limits(limits=1)
SEED = 42
DATA = Path("data")
RUN_VALIDATION = True
REPRODUCE_PREVIOUS = False  # v4, v5 и v7; ещё 12 обучений
TEST_DAY = pd.Timestamp("2026-04-20")
np.random.seed(SEED)
print("Python", platform.python_version(), "NumPy", np.__version__,
      "pandas", pd.__version__, "scikit-learn", sklearn.__version__)


Python 3.12.14 NumPy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0


## 1. Очистка событий

- Оставляю только события внутри окна куки `[window_start_ts, window_end_ts)`. В train около 17% событий лежат после окна, включая все `captcha_shown`; в test их нет.
- Удаляю полные дубли строк.
- Привожу написание платформы к трём значениям.
- Стабильно сортирую по времени. Порядок внутри одной секунды не восстанавливается, поэтому признаки переходов его не используют.

Ниже — семейства User-Agent: сырая строка в модель не идёт, только грубое семейство клиента.

In [2]:
# Фиксированные категории: набор признаков не зависит от состава test.
META_TIME_COLS = ["cookie_created_at", "window_start_ts", "window_end_ts"]
RAW_EVENT_COLS = [
    "cookie_id", "event_ts", "eid", "event_name", "platform", "user_agent",
    "item_id", "item_category", "item_location", "seller_type",
    "search_query", "search_page", "pointer_x", "pointer_y",
]
EVENT_TYPES = [
    "search_results_view", "item_view", "photo_swipe", "seller_page_view",
    "contact_phone_show", "contact_chat_open", "contact_message_sent",
    "favorite_add", "login",
]
PLATFORM_MAP = {"web": "web", "desktop": "web", "android": "android",
                "ios": "ios", "iphone": "ios"}
def normalize_platform(platform: pd.Series) -> pd.Series:
    """Приводим варианты написания платформы к одному значению."""
    return platform.astype("string").str.strip().str.lower().map(PLATFORM_MAP).fillna("unknown").astype(str)

def _check_meta(meta: pd.DataFrame) -> None:
    """Одна строка на куку, непустые даты и положительная длина окна."""
    missing = {"cookie_id", *META_TIME_COLS} - set(meta.columns)
    if missing:
        raise ValueError(f"в meta нет колонок {sorted(missing)}")
    if not meta["cookie_id"].is_unique:
        raise ValueError("cookie_id в meta должен быть уникальным (одно окно на куку)")
    if meta[META_TIME_COLS].isna().any().any():
        raise ValueError("в meta есть пустые даты")
    if not (meta["window_end_ts"] > meta["window_start_ts"]).all():
        raise ValueError("есть окна нулевой или отрицательной длины")

def prepare_events(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Оставляем [начало, конец), удаляем полные дубли и стабильно сортируем события."""
    missing = set(RAW_EVENT_COLS) - set(events.columns)
    if missing:
        raise ValueError(f"ожидаются сырые события, нет колонок {sorted(missing)}")
    _check_meta(meta)
    if events["event_ts"].isna().any():
        raise ValueError("в events есть пустые event_ts")

    ev = events[RAW_EVENT_COLS].merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id", how="inner", validate="many_to_one",   # куки вне meta намеренно отбрасываются
    )
    in_window = (ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])
    ev = ev[in_window].drop_duplicates(subset=RAW_EVENT_COLS)
    ev = ev.sort_values(["cookie_id", "event_ts", "eid"], kind="mergesort").reset_index(drop=True)
    ev["plat"] = normalize_platform(ev["platform"])
    return ev


In [3]:
def ua_family(user_agent) -> str:
    """Семейство клиента. Headless проверяется до Chrome, приложение — до ОС."""
    if not isinstance(user_agent, str) or not user_agent.strip():
        return "unknown"
    ua = user_agent.lower()
    if "headlesschrome" in ua:
        return "headless"
    if any(t in ua for t in ("python", "curl", "go-http", "scrapy", "node-fetch")):
        return "script"
    if ua.startswith("avito/"):
        if "android" in ua:
            return "avito_app_android"
        if "iphone" in ua or "ios" in ua:
            return "avito_app_ios"
        return "unknown"
    if "iphone" in ua:
        return "mobile_safari" if "crios" not in ua and "fxios" not in ua else "mobile_other"
    if "android" in ua:
        return "mobile_chrome" if "chrome" in ua and "firefox" not in ua else "mobile_other"
    if "yabrowser" in ua:
        return "yabrowser"
    if "firefox" in ua:
        return "firefox"
    if "chrome" in ua:
        return "chrome"
    if "safari" in ua:
        return "safari"
    return "unknown"


## 2. Загрузка данных

Проверяю размеры и то, что train и test не пересекаются по `cookie_id`.

In [4]:
train = pd.read_csv(DATA / "train.csv", parse_dates=META_TIME_COLS)
test = pd.read_csv(DATA / "test.csv", parse_dates=META_TIME_COLS)
events_path = DATA / "events.csv.gz"
if not events_path.exists():
    events_path = DATA / "events.csv"
events = pd.read_csv(events_path, parse_dates=["event_ts"])

assert len(train) == 11091 and len(test) == 4909
assert train.target.isin([0, 1]).all()
assert not train.cookie_id.isin(test.cookie_id).any()
meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
labels = train.set_index("cookie_id").target
train_ids = train.cookie_id.to_numpy()
test_ids = test.cookie_id.to_numpy()
print(f"Train: {len(train)} кук, {labels.sum()} ботов; test: {len(test)} кук")


Train: 11091 кук, 899 ботов; test: 4909 кук


## 3. Признаки поведения куки (130)

Каждая группа считается по событиям одной куки, метки здесь не используются.

Правило для пропусков:
- счётчики при отсутствии наблюдений равны 0;
- доли, квантили и энтропии, которые нельзя определить, — NaN. HistGradientBoosting работает с NaN напрямую.

In [5]:
EPOCH = pd.Timestamp("2026-01-01")
GAP_EDGES = [-0.5, 0, 12, 60, 360, 900, 1800, np.inf]          # 0 | (0,12] | (12,60] | ... | >1800 с
GAP_LABELS = ["0", "0_12", "12_60", "60_360", "360_900", "900_1800", "gt1800"]
GAP_QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
SESSION_GAP_S = 1800                                             # сессия: пауза не больше 30 минут
DEEP_PAGE = 5                                                    # «глубокая» страница выдачи
POINTER_X_LIMITS, POINTER_Y_LIMITS = (0, 1920), (0, 1080)        # наблюдаемые пределы координат в данных
UA_FAMILIES = ["chrome", "firefox", "yabrowser", "safari", "headless", "script",
               "avito_app_android", "avito_app_ios", "mobile_chrome", "mobile_safari", "mobile_other", "unknown"]
PLATFORMS = ["web", "android", "ios"]

def _div(a: pd.Series, b: pd.Series) -> pd.Series:
    """Доля не определена при нулевом знаменателе."""
    return a / b.where(b > 0)

def _mean_by_cookie(cookie_id: pd.Series, flag: pd.Series) -> pd.Series:
    """Доля True среди событий каждой куки."""
    return flag.astype(float).groupby(cookie_id.values).mean()

def _concentration(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    """Число значений, уникальные значения, доля самого частого и энтропия."""
    d = df[["cookie_id", col]].dropna()
    cnt = d.groupby(["cookie_id", col]).size()
    total = cnt.groupby(level=0).sum()
    p = cnt / total.reindex(cnt.index.get_level_values(0)).to_numpy()
    return pd.DataFrame({
        f"{prefix}_n_known": total,
        f"{prefix}_nunique": cnt.groupby(level=0).size(),
        f"{prefix}_top_share": cnt.groupby(level=0).max() / total,
        f"{prefix}_entropy": -(p * np.log(p)).groupby(level=0).sum(),
    })

def _max_events_in_window(ev: pd.DataFrame, seconds: int) -> pd.Series:
    """Максимум событий в [t, t + seconds); вход отсортирован по куке и времени."""
    code = ev["cookie_id"].astype("category").cat.codes.to_numpy(np.int64)   # коды монотонны вместе с cookie_id
    key = code * 10**9 + ev["t"].to_numpy(np.int64)                         # t < 10^9 секунд, куки не смешиваются
    cnt = np.searchsorted(key, key + seconds, side="left") - np.arange(len(key))
    return pd.Series(cnt, index=ev["cookie_id"].to_numpy()).groupby(level=0).max()

def _base(ev: pd.DataFrame) -> pd.DataFrame:
    """Индекс всех кук с событиями, чтобы не терять их при объединении агрегатов."""
    return pd.DataFrame(index=pd.Index(ev["cookie_id"].unique(), name="cookie_id"))


In [6]:
def rhythm_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Темп и ритм: паузы между событиями, короткие паузы, плотность, время суток, сессии."""
    f = _base(ev)
    g = ev.groupby("cookie_id")
    gaps = ev.loc[ev["gap"].notna(), ["cookie_id", "gap"]]
    gg = gaps.groupby("cookie_id")["gap"]

    f["n_events"] = g.size()
    f["n_unique_seconds"] = g["t"].nunique()
    f["n_gaps"] = gg.size()
    # Квантили пауз. Колонки задаются явно: у набора без пауз их не будет.
    q = {k: gg.quantile(k) for k in GAP_QUANTILES} if len(gaps) else {k: pd.Series(dtype=float) for k in GAP_QUANTILES}
    f["gap_p10"], f["gap_median"], f["gap_p90"] = q[0.1], q[0.5], q[0.9]
    f["gap_iqr"] = q[0.75] - q[0.25]
    f["gap_mean"] = gg.mean()
    f["gap_std"] = gg.std(ddof=1)
    # Доли пауз по непересекающимся диапазонам. Знаменатель — наблюдаемые паузы (без первого события).
    bins = pd.cut(gaps["gap"], GAP_EDGES, labels=GAP_LABELS)
    shares = (pd.get_dummies(bins).astype(float).groupby(gaps["cookie_id"].to_numpy()).mean()
              .reindex(columns=GAP_LABELS, fill_value=0.0))
    for lbl in GAP_LABELS:
        f[f"gap_share_{lbl}"] = shares[lbl]
    f["gap_share_le12"] = f["gap_share_0"] + f["gap_share_0_12"]

    # Ритм внутри активных периодов: только паузы ≤ 30 минут (длинные перерывы смешивают картину)
    short = gaps[gaps["gap"] <= SESSION_GAP_S].groupby("cookie_id")["gap"]
    f["n_short_gaps"] = short.size()
    f["short_gap_median"] = short.median()
    f["short_gap_mean"] = short.mean()
    f["short_gap_std"] = short.std(ddof=1)
    f["short_gap_cv"] = _div(f["short_gap_std"], f["short_gap_mean"])

    # Плотность и распределение по времени
    f["active_span_s"] = g["t"].max() - g["t"].min()
    f["max_events_60s"] = _max_events_in_window(ev, 60)
    f["max_events_300s"] = _max_events_in_window(ev, 300)
    f["n_hours_active"] = g["hour"].nunique()
    f["share_hours_0_6"] = _mean_by_cookie(ev["cookie_id"], ev["hour"] < 6)   # часы по исходным timestamp

    # Сессии (пауза > 30 минут): концентрация активности в одной сессии
    sess_size = ev.groupby(["cookie_id", "session"]).size()
    f["n_sessions"] = sess_size.groupby(level=0).size()
    f["max_session_share"] = sess_size.groupby(level=0).max() / f["n_events"]
    return f


In [7]:
def _query_transitions(s: pd.DataFrame, within_session: bool) -> pd.DataFrame:
    """Соседние поиски с одним запросом; пары с одинаковым временем исключены."""
    tied = s.duplicated(["cookie_id", "t"], keep=False)
    prev = s.shift()
    ok = ((s["cookie_id"] == prev["cookie_id"]) & ~tied & ~tied.shift(fill_value=True)
          & s["search_query"].notna() & (s["search_query"] == prev["search_query"])
          & s["search_page"].notna() & prev["search_page"].notna())
    if within_session:
        ok &= s["session"] == prev["session"]
    step = s["search_page"] - prev["search_page"]
    t = pd.DataFrame({"ok": ok, "next": ok & (step == 1), "back": ok & (step < 0), "stay": ok & (step == 0)})
    return t.groupby(s["cookie_id"].to_numpy()).sum()

def search_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Обход выдачи: глубина страниц, повтор запросов, последовательное листание."""
    f = _base(ev)
    s = ev[ev["event_name"] == "search_results_view"]
    gs = s.groupby("cookie_id")
    page = s.dropna(subset=["search_page"])
    f["n_search"] = gs.size()
    f["n_search_with_query"] = gs["search_query"].count()
    f["n_unique_queries"] = gs["search_query"].nunique()
    f["page_mean"] = gs["search_page"].mean()
    f["page_median"] = gs["search_page"].median()
    f["page_max"] = gs["search_page"].max()
    f["page_first_share"] = _mean_by_cookie(page["cookie_id"], page["search_page"] == 1)
    f["page_deep_share"] = _mean_by_cookie(page["cookie_id"], page["search_page"] >= DEEP_PAGE)
    f["query_duplicate_fraction"] = 1 - _div(f["n_unique_queries"], f["n_search_with_query"])
    qc = _concentration(s, "search_query", "query")
    f["query_top_share"], f["query_entropy"] = qc["query_top_share"], qc["query_entropy"]

    qp = s.dropna(subset=["search_query", "search_page"])
    n_qp = qp.groupby("cookie_id").size()
    n_qp_unique = qp.drop_duplicates(["cookie_id", "search_query", "search_page"]).groupby("cookie_id").size()
    f["repeat_query_page_fraction"] = 1 - _div(n_qp_unique.reindex(f.index), n_qp.reindex(f.index))
    f["max_pages_per_query"] = qp.groupby(["cookie_id", "search_query"])["search_page"].nunique().groupby(level=0).max()

    # Соседние поиски с одним запросом, в том числе через паузу > 30 минут (разные сессии).
    # Семантика: «вернулась ли кука к тому же запросу и на какую страницу», а не только листание в одной сессии.
    tr = _query_transitions(s, within_session=False)
    f["n_same_query_steps"] = tr["ok"]
    f["same_query_next_page_share"] = _div(tr["next"], tr["ok"])
    f["same_query_page_back_share"] = _div(tr["back"], tr["ok"])
    f["same_query_same_page_share"] = _div(tr["stay"], tr["ok"])
    return f


In [8]:
def diversity_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Широта и концентрация интересов: объявления, категории, города, продавцы."""
    f = _base(ev)
    iv = ev[ev["event_name"] == "item_view"]
    with_item = ev[ev["item_id"].notna()]
    giv = iv.groupby("cookie_id")
    f["n_item_views"] = giv.size()
    f["n_unique_items_viewed"] = giv["item_id"].nunique()
    f["n_unique_items_all"] = with_item.groupby("cookie_id")["item_id"].nunique()
    f["item_view_repeat_fraction"] = 1 - _div(f["n_unique_items_viewed"], f["n_item_views"])
    # Категории и города просмотренных объявлений (photo_swipe не раздувает интерес к одному объявлению)
    f = f.join(_concentration(iv, "item_category", "view_cat")).join(_concentration(iv, "item_location", "view_loc"))
    f["view_loc_per_known"] = _div(f["view_loc_nunique"], f["view_loc_n_known"])
    # По всем событиям, включая фильтры поиска: широта обхода по регионам и категориям
    f = f.join(_concentration(ev, "item_location", "all_loc")[["all_loc_nunique", "all_loc_top_share"]])
    f = f.join(_concentration(ev, "item_category", "all_cat")[["all_cat_nunique", "all_cat_top_share"]])
    seller = with_item.dropna(subset=["seller_type"])
    f["seller_pro_share"] = _mean_by_cookie(seller["cookie_id"], seller["seller_type"] == "pro")
    return f


In [9]:
def composition_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Состав действий: счётчики и доли типов событий, отношения между ними."""
    cnt = (pd.get_dummies(ev["event_name"]).groupby(ev["cookie_id"].to_numpy()).sum()
           .reindex(index=_base(ev).index, columns=EVENT_TYPES, fill_value=0).astype(float))
    n = cnt.sum(axis=1)
    f = cnt.add_prefix("cnt_").join(cnt.div(n.where(n > 0), axis=0).add_prefix("share_"))
    f["n_event_types"] = (cnt > 0).sum(axis=1)
    contacts = cnt[["contact_phone_show", "contact_chat_open", "contact_message_sent"]].sum(axis=1)
    f["cnt_contacts"] = contacts
    views = cnt["item_view"]
    f["photo_per_view"] = _div(cnt["photo_swipe"], views)
    f["contact_per_view"] = _div(contacts, views)
    f["favorite_per_view"] = _div(cnt["favorite_add"], views)
    f["seller_page_per_view"] = _div(cnt["seller_page_view"], views)
    f["view_per_search"] = _div(views, cnt["search_results_view"])
    return f


In [10]:
def pointer_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Курсор web-куки: наличие, разброс, повторы и доли граничных/округлённых координат."""
    f = _base(ev)
    web = ev[ev["plat"] == "web"]
    p = web.dropna(subset=["pointer_x", "pointer_y"])
    gp = p.groupby("cookie_id")
    f["n_web_events"] = web.groupby("cookie_id").size().reindex(f.index, fill_value=0)
    f["n_pointer"] = gp.size().reindex(f.index, fill_value=0)
    f["pointer_share"] = _div(f["n_pointer"], f["n_web_events"])
    for c in ("pointer_x", "pointer_y"):
        f[f"{c}_mean"] = gp[c].mean()
        f[f"{c}_median"] = gp[c].median()
        f[f"{c}_std"] = gp[c].std(ddof=1)
    pairs = p.groupby(["cookie_id", "pointer_x", "pointer_y"]).size()
    f["pointer_unique_pair_share"] = _div(pairs.groupby(level=0).size().reindex(f.index), f["n_pointer"])
    f["pointer_top_pair_share"] = _div(pairs.groupby(level=0).max().reindex(f.index), f["n_pointer"])
    inner_x = ~p["pointer_x"].isin(POINTER_X_LIMITS)
    inner_y = ~p["pointer_y"].isin(POINTER_Y_LIMITS)
    # Знаменатель всех трёх долей — все пары координат куки. round50_* — совместная доля
    # «значение вне пределов и кратно 50», а не условная кратность среди внутренних значений.
    f["pointer_limit_share"] = _mean_by_cookie(p["cookie_id"], ~inner_x | ~inner_y)
    f["pointer_round50_x_share"] = _mean_by_cookie(p["cookie_id"], inner_x & (p["pointer_x"] % 50 == 0))
    f["pointer_round50_y_share"] = _mean_by_cookie(p["cookie_id"], inner_y & (p["pointer_y"] % 50 == 0))
    return f


In [11]:
def client_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Платформа и семейство клиента по User-Agent (сырая строка UA в модель не идёт)."""
    f = _base(ev)
    fam_map = {u: ua_family(u) for u in ev["user_agent"].dropna().unique()}
    fam = ev["user_agent"].map(fam_map).fillna("unknown")
    cid = ev["cookie_id"].to_numpy()
    f["n_platforms"] = ev.groupby("cookie_id")["plat"].nunique()
    f["n_user_agents"] = ev.groupby("cookie_id")["user_agent"].nunique()
    f["n_ua_families"] = fam.groupby(cid).nunique()
    plat_share = (pd.get_dummies(ev["plat"]).astype(float).groupby(cid).mean()
                  .reindex(columns=PLATFORMS, fill_value=0.0))
    fam_share = (pd.get_dummies(fam).astype(float).groupby(cid).mean()
                 .reindex(columns=UA_FAMILIES, fill_value=0.0))
    return f.join(plat_share.add_prefix("plat_share_")).join(fam_share.add_prefix("ua_share_"))

def cookie_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Возраст куки и положение активности внутри окна."""
    m = meta.set_index("cookie_id")
    g = ev.groupby("cookie_id")["event_ts"]
    window_s = (m["window_end_ts"] - m["window_start_ts"]).dt.total_seconds()
    first, last = g.min().reindex(m.index), g.max().reindex(m.index)
    f = pd.DataFrame(index=m.index)
    f["cookie_age_days"] = (m["window_start_ts"] - m["cookie_created_at"]).dt.total_seconds() / 86400
    f["first_event_offset_h"] = (first - m["window_start_ts"]).dt.total_seconds() / 3600
    f["last_event_to_end_h"] = (m["window_end_ts"] - last).dt.total_seconds() / 3600
    f["active_window_share"] = (last - first).dt.total_seconds() / window_s
    return f


In [12]:
def build_features(events, meta):
    """130 признаков в фиксированном порядке; target здесь не используется."""
    ev = prepare_events(events, meta)
    ev["t"] = (ev["event_ts"] - EPOCH).dt.total_seconds().astype(np.int64)
    ev["gap"] = ev.groupby("cookie_id")["t"].diff()
    ev["hour"] = ev["event_ts"].dt.hour
    ev["session"] = (ev["gap"].isna() | (ev["gap"] > SESSION_GAP_S)).groupby(ev["cookie_id"]).cumsum()
    index = pd.Index(meta["cookie_id"], name="cookie_id")
    builders = [rhythm_features, search_features, diversity_features,
                composition_features, pointer_features, client_features]
    parts = [builder(ev).reindex(index) for builder in builders]
    parts.append(cookie_features(ev, meta).reindex(index))
    x = pd.concat(parts, axis=1)

    # Отсутствие событий — нулевой счётчик; неопределённая статистика — NaN.
    counts = [c for c in x if c.startswith(("n_", "cnt_", "max_events_"))
              or c.endswith(("_n_known", "_nunique"))]
    x[counts] = x[counts].fillna(0)
    flags = pd.DataFrame({
        "has_gaps": x["n_gaps"] > 0,
        "has_search": x["n_search"] > 0,
        "has_item_view": x["n_item_views"] > 0,
        "has_web": x["n_web_events"] > 0,
        "has_pointer": x["n_pointer"] > 0,
    }, index=index).astype(np.int8)
    x = x.join(flags).astype(np.float64)
    assert x.shape == (len(meta), 130) and x.index.equals(index)
    assert not np.isinf(x.to_numpy()).any()
    pairs = ev.loc[ev.item_id.notna(), ["cookie_id", "item_id"]].drop_duplicates()
    return x, pairs


## 4. Признаки по истории: пул объявлений ботов

Все три функции получают только историю — размеченные train-куки, чьи окна закончились до момента предсказания. Своя метка и метки валидации сюда не попадают.

- **V3** (`pool_v3`) — доля и число объявлений куки, которые смотрели прошлые боты, и сглаженная доля ботов среди прошлых зрителей объявления: `(ботов + 2·0.08) / (зрителей + 2)`.
- **CP** (`pool_cp`) — то же, но по «очищенному» пулу: только объявления ботов, у которых ≥30% объявлений встречались у других ботов, и не меньше 3 объявлений. Сюда же входит признак правдоподобия `cp_llr` по попаданиям и промахам.
- `pool_raw_llr` — тот же LLR по неочищенному пулу. Нужен только для воспроизведения версии v5.

In [13]:
# Сглаживание одинаковое для V3 и CP.
PRIOR, SMOOTH = 0.08, 2.0
POOL_COLUMNS = ["wl_share", "wl_n", "wl_te_mean", "wl_te_max"]
CP_COLUMNS = ["cp_share", "cp_n", "cp_te_mean", "cp_te_max", "cp_llr"]
def pool_v3(pairs, target, history, labels):
    """Четыре агрегата V3 по уникальным парам (кука, объявление)."""
    if len(pd.Index(target).intersection(history)):
        raise ValueError("Собственная метка не может входить в историю")
    h = pairs[pairs.cookie_id.isin(history)].copy()
    h["t"] = labels.reindex(h.cookie_id).to_numpy()
    st = h.groupby("item_id").agg(n=("cookie_id", "size"), b=("t", "sum"))
    c = pairs[pairs.cookie_id.isin(target)].join(st, on="item_id")
    c[["b", "n"]] = c[["b", "n"]].fillna(0)
    c["in_pool"] = c.b > 0
    c["te"] = (c.b + SMOOTH * PRIOR) / (c.n + SMOOTH)
    g = c.groupby("cookie_id")
    f = pd.DataFrame({"wl_share": g.in_pool.mean(), "wl_n": g.in_pool.sum(),
                      "wl_te_mean": g.te.mean(), "wl_te_max": g.te.max()})
    f = f.reindex(pd.Index(target, name="cookie_id"))
    f["wl_n"] = f.wl_n.fillna(0)
    return f[POOL_COLUMNS].astype(np.float64)


In [14]:
def pool_cp(pairs, cur, hist, lab, tau=0.3, min_items=3):
    # Пуловые боты: ≥3 объявления и ≥30% пересекаются с другими ботами истории.
    h = pairs[pairs.cookie_id.isin(hist)]
    yb = lab.reindex(h.cookie_id).to_numpy()
    bots = h[yb == 1]
    n_all = h.groupby("item_id").size()
    nb_item = bots.groupby("item_id").size()
    other = bots.item_id.map(nb_item).to_numpy() - 1 > 0
    loo = pd.Series(other, index=bots.cookie_id.to_numpy()).groupby(level=0).mean()
    kb = bots.groupby("cookie_id").size()
    pool_bots = loo.index[(loo >= tau) & (kb.reindex(loo.index) >= min_items)]
    pb = bots[bots.cookie_id.isin(pool_bots)]
    b_pool = pb.groupby("item_id").size()
    # доли попаданий в очищенный пул: пуловые боты (без себя) и люди истории
    if len(pb):
        p_b = float((pb.item_id.map(b_pool).to_numpy() - 1 > 0).mean())
    else:
        p_b = np.nan
    hum = h[yb == 0]
    p_h = float(hum.item_id.isin(b_pool.index).mean()) if len(hum) else np.nan
    c = pairs[pairs.cookie_id.isin(cur)]
    bp = c.item_id.map(b_pool).fillna(0).to_numpy()
    n = c.item_id.map(n_all).fillna(0).to_numpy()
    d = pd.DataFrame({"in": bp > 0, "te": (bp + SMOOTH * PRIOR) / (n + SMOOTH)}, index=c.cookie_id.to_numpy())
    g = d.groupby(level=0)
    f = pd.DataFrame({"cp_share": g["in"].mean(), "cp_n": g["in"].sum(),
                      "cp_te_mean": g["te"].mean(), "cp_te_max": g["te"].max()})
    f = f.reindex(cur)
    f["cp_n"] = f.cp_n.fillna(0)
    # h = cp_n попаданий из k объявлений; вероятности оценены только на истории.
    k = c.groupby("cookie_id").size().reindex(cur)
    # Без объявлений/нужного класса истории LLR остаётся NaN. Clipping защищает log.
    eps = 1e-6
    if np.isfinite(p_b) and np.isfinite(p_h):
        pb_, ph_ = np.clip(p_b, eps, 1 - eps), np.clip(p_h, eps, 1 - eps)
        f["cp_llr"] = f.cp_n * np.log(pb_ / ph_) + (k - f.cp_n) * np.log((1 - pb_) / (1 - ph_))
    else:
        f["cp_llr"] = np.nan
    return f


In [15]:
def pool_raw_llr(pairs, cur, hist, lab):
    """Вариант v5: тот же LLR по объявлениям всех исторических ботов."""
    h = pairs[pairs.cookie_id.isin(hist)]
    yb = lab.reindex(h.cookie_id).to_numpy()
    bots = h[yb == 1]
    nb_item = bots.groupby("item_id").size()
    p_b = float((bots.item_id.map(nb_item).to_numpy() - 1 > 0).mean()) if len(bots) else np.nan
    hum = h[yb == 0]
    p_h = float(hum.item_id.isin(nb_item.index).mean()) if len(hum) else np.nan
    c = pairs[pairs.cookie_id.isin(cur)]
    hit = pd.Series(c.item_id.isin(nb_item.index).to_numpy(), index=c.cookie_id.to_numpy()).groupby(level=0)
    hh, k = hit.sum().reindex(cur), hit.size().reindex(cur)
    f = pd.DataFrame(index=cur)
    eps = 1e-6
    if np.isfinite(p_b) and np.isfinite(p_h):
        pb_, ph_ = np.clip(p_b, eps, 1 - eps), np.clip(p_h, eps, 1 - eps)
        f["raw_llr"] = hh * np.log(pb_ / ph_) + (k - hh) * np.log((1 - pb_) / (1 - ph_))
    else:
        f["raw_llr"] = np.nan
    return f


## 5. Сборка матрицы признаков без утечек

`FeatureBuilder` хранит события, но не метки: метки обучающей части передаются явно в каждом вызове.

- **Обучающая строка дня d** видит историю за 7 дней до d.
- **Валидационный блок и test** получают одну историю, замороженную на начало блока: второй день блока не видит меток первого. Для test это train-окна 13–19 апреля, `ref_day = 2026-04-20`.
- Проверки `assert` не дают передать лишние метки и перепутать train с validation.

In [16]:
class FeatureBuilder:
    def __init__(self, events, meta):
        # В объекте нет меток: они передаются отдельно для обучающей части фолда.
        meta = meta[["cookie_id", *META_TIME_COLS]].copy()
        self.base, self.pairs = build_features(events, meta)
        self.meta = meta.set_index("cookie_id")
        self.day = self.meta.window_start_ts.dt.normalize()

    def history(self, allowed, cutoff, window_days=7):
        h = self.meta.loc[allowed]
        mask = (h.window_start_ts < cutoff) & (h.window_end_ts <= cutoff)
        if window_days is not None:
            mask &= h.window_start_ts >= cutoff - pd.Timedelta(days=window_days)
        return h.index[mask]

    def history_features(self, fn, columns, target, allowed, labels, ref_day, window_days):
        target = pd.Index(target, name="cookie_id")
        allowed = pd.Index(allowed, name="cookie_id")
        assert target.is_unique and allowed.is_unique and labels.index.is_unique
        assert set(labels.index) == set(allowed), "Передайте только train-метки фолда"
        assert labels.isin([0, 1]).all()
        if ref_day is None:
            days = self.day.loc[target]
            groups = days.groupby(days).groups.items()
        else:
            ref_day = pd.Timestamp(ref_day)
            assert not target.isin(allowed).any()
            assert (self.meta.loc[target, "window_start_ts"] >= ref_day).all()
            assert (self.meta.loc[allowed, "window_end_ts"] <= ref_day).all()
            groups = [(ref_day, target)]  # одна история для всего блока, включая второй день

        parts = []
        for cutoff, current in groups:
            history = self.history(allowed, cutoff, window_days)
            assert not pd.Index(current).isin(history).any()
            parts.append(fn(self.pairs, current, history, labels.loc[history]))
        return pd.concat(parts).reindex(target)[columns].astype(np.float64)

    def matrix(self, target, allowed, labels, ref_day=None, variant="v6"):
        assert variant in ("v4", "v5", "v6", "v7")
        v3 = self.history_features(pool_v3, POOL_COLUMNS, target, allowed, labels, ref_day, 7)
        x = self.base.loc[v3.index].join(v3)
        if variant == "v5":
            extra = self.history_features(pool_raw_llr, ["raw_llr"], target, allowed, labels, ref_day, 7)
            x = x.join(extra)
        elif variant in ("v6", "v7"):
            # В v7 меняется только окно CP; V3 всегда использует 7 дней.
            window = 7 if variant == "v6" else None
            extra = self.history_features(pool_cp, CP_COLUMNS, target, allowed, labels, ref_day, window)
            x = x.join(extra)
        x = x.astype(np.float64)
        assert not np.isinf(x.to_numpy()).any()
        return x


builder = FeatureBuilder(events, meta)
print("Поведенческие признаки:", builder.base.shape)


Поведенческие признаки: (16000, 130)


## 6. Модель и метрика

Для v4–v6 параметры модели одинаковы, подбора под test не было; в v7 использовано усреднение десяти моделей с `max_features=0.8`. Метрика — копия официальной: максимальная precision среди порогов с recall ≥ 0.70, куки с одинаковым score не разделяются.

In [17]:
def make_model(seed=SEED, max_features=1.0):
    return HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=20, l2_regularization=1.0,
        early_stopping=False, random_state=seed, max_features=max_features,
    )


TARGET_RECALL = 0.70
def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # последняя строка каждой группы равных score
    return tp[ends] / k[ends], tp[ends] / n_pos

def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


# Проверка официального определения: максимум по всем допустимым порогам, ties вместе.
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5


## 7. Валидация: v4 (без очищенного пула) против v6 (итог)

Четыре последовательных двухдневных блока и семидневный блок 13–19 апреля. Модель каждый раз учится только на окнах, закончившихся до начала блока, а признаки истории пересчитываются внутри фолда. Обе версии сравниваются на одних и тех же куках.

In [18]:
# Baseline и итоговая модель оцениваются на одних и тех же куках.
FOLDS = [
    ("time0", "2026-04-12", "2026-04-14"),
    ("time1", "2026-04-14", "2026-04-16"),
    ("time2", "2026-04-16", "2026-04-18"),
    ("time3", "2026-04-18", "2026-04-20"),
    ("block7", "2026-04-13", "2026-04-20"),
]
validation_rows = []
validation_predictions = {}
if RUN_VALIDATION:
    for fold, start, end in FOLDS:
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        fit_ids = train.loc[train.window_end_ts <= start, "cookie_id"].to_numpy()
        valid_ids = train.loc[(train.window_start_ts >= start) &
                             (train.window_start_ts < end), "cookie_id"].to_numpy()
        fit_labels = labels.loc[fit_ids]
        assert not np.isin(fit_ids, valid_ids).any()
        x_fit = builder.matrix(fit_ids, fit_ids, fit_labels)
        x_valid = builder.matrix(valid_ids, fit_ids, fit_labels, start)
        for variant in ("v4", "v6"):
            columns = x_fit.columns[:134] if variant == "v4" else x_fit.columns
            model = make_model().fit(x_fit[columns], fit_labels.loc[x_fit.index])
            score = model.predict_proba(x_valid[columns])[:, 1]
            y = labels.loc[valid_ids].to_numpy()
            validation_predictions[variant, fold] = pd.DataFrame({
                "cookie_id": valid_ids, "target": y, "score": score})
            validation_rows.append({"variant": variant, "fold": fold,
                                    "P@R70": precision_at_recall(y, score),
                                    "AP": average_precision_score(y, score)})
        print(f"{fold}: train={len(fit_ids)}, validation={len(valid_ids)}")

    metrics = pd.DataFrame(validation_rows)
    print(metrics.pivot(index="fold", columns="variant", values=["P@R70", "AP"]).round(6).to_string())
    temporal = metrics[metrics.fold.str.startswith("time")]
    summary = temporal.groupby("variant").agg(
        temporal_mean=("P@R70", "mean"), worst_fold=("P@R70", "min"), mean_AP=("AP", "mean"))
    print("\nTemporal summary:\n" + summary.round(6).to_string())


time0: train=5113, validation=1660
time1: train=6773, validation=1627
time2: train=8400, validation=1453
time3: train=9853, validation=1238
block7: train=5930, validation=5161
            P@R70                  AP          
variant        v4        v6        v4        v6
fold                                           
block7   0.877676  0.902208  0.827657  0.833723
time0    0.893617  0.894737  0.797122  0.801902
time1    0.904762  0.950495  0.847173  0.860107
time2    0.898990  0.902913  0.845965  0.850059
time3    0.870130  0.847059  0.836372  0.837052

Temporal summary:
         temporal_mean  worst_fold   mean_AP
variant                                     
v4            0.891875    0.870130  0.831658
v6            0.898801    0.847059  0.837280


## 8. Финальное обучение и submission

Модель учится на всём train. Test получает историю train за 13–19 апреля.

Ниже проверяется, что признаки и score test-куки не меняются, если убрать остальные test-куки. Потом `submission.csv` записывается с проверкой формата и SHA-256 отправленного файла.

In [19]:
# Финальное обучение: train-строки видят своё прошлое, test — историю до 20 апреля.
x_train = builder.matrix(train_ids, train_ids, labels)
x_test = builder.matrix(test_ids, train_ids, labels, TEST_DAY)
assert x_train.shape[1] == x_test.shape[1] == 139
history = builder.history(pd.Index(train_ids), TEST_DAY, 7)
assert builder.meta.loc[history, "window_start_ts"].min() == pd.Timestamp("2026-04-13")
assert builder.meta.loc[history, "window_end_ts"].max() == TEST_DAY
model = make_model().fit(x_train, labels.loc[x_train.index])
scores = model.predict_proba(x_test)[:, 1]
print(f"Признаков: {x_train.shape[1]}; куки в истории test: {len(history)}")


Признаков: 139; куки в истории test: 5161


In [20]:
# Пересчитываем признаки после удаления остальных test-кук, без повторного обучения.
sample_ids = test_ids[::97]
small_meta = meta[meta.cookie_id.isin(np.r_[train_ids, sample_ids])]
small_events = events[events.cookie_id.isin(small_meta.cookie_id)]
small_builder = FeatureBuilder(small_events, small_meta)
small_x = small_builder.matrix(sample_ids, train_ids, labels, TEST_DAY)
pd.testing.assert_frame_equal(small_x, x_test.loc[sample_ids])
np.testing.assert_array_equal(
    model.predict_proba(small_x)[:, 1], pd.Series(scores, index=test_ids).loc[sample_ids].to_numpy())
del small_builder, small_events
print("Независимость от состава test-батча: OK")


Независимость от состава test-батча: OK


In [21]:
EXPECTED_SHA = {
    "v4": "6f423457d9de8ca7f2ca6547b58eb52161168c56193c62a9b86e048fb00da40a",
    "v5": "4037d21a5c6950d7784fcd9e528c5b163f2571e11e53e5fa8dd8892da26ed4e7",
    "v6": "9056acbaffed9d4605f0d09caf8d0bac9ddbadd6cb466fa54a3b7b52c8d537e3",
    "v7": "3a4bcd837d81ee721151028cd8ae26d32e3dd13408a643294d9f61e9ffb64de4",
}


def write_submission(scores, path, variant):
    result = pd.DataFrame({"cookie_id": test_ids, "score": np.asarray(scores, dtype=float)})
    assert list(result.columns) == ["cookie_id", "score"] and len(result) == 4909
    assert result.cookie_id.is_unique and result.cookie_id.equals(test.cookie_id)
    assert np.isfinite(result.score).all() and result.score.between(0, 1).all()
    assert result.score.nunique() > 1000, "Ожидаются непрерывные вероятности"
    content = result.to_csv(index=False, lineterminator="\n").encode("utf-8")
    digest = hashlib.sha256(content).hexdigest()
    # Проверяется именно результат обучения. Готовый CSV не читается и не используется моделью.
    path = Path(path)
    if digest != EXPECTED_SHA[variant]:
        # Другая ОС или версии библиотек могут изменить последние знаки float.
        # Отправленный файл не перезаписываем, новый результат кладём рядом.
        path = path.with_name(path.stem + "_reproduced.csv")
        print(f"ВНИМАНИЕ: SHA-256 {digest} отличается от отправленного {EXPECTED_SHA[variant]}.\n"
              f"Проверьте версии из requirements.txt. Результат сохранён в {path}.")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)
    print(f"{path}: {len(result)} строк; SHA-256 {digest}"
          + (" — совпадает с отправленным файлом" if digest == EXPECTED_SHA[variant] else ""))
    return result


submission = write_submission(scores, "submission.csv", "v6")


submission.csv: 4909 строк; SHA-256 9056acbaffed9d4605f0d09caf8d0bac9ddbadd6cb466fa54a3b7b52c8d537e3 — совпадает с отправленным файлом


## 9. (Необязательно) воспроизведение прежних отправок

При `REPRODUCE_PREVIOUS = True` создаёт в `previous/` версии v4, v5 и v7 и сверяет их SHA-256 с отправленными файлами. Основной `submission.csv` не меняется.

In [22]:
# Необязательно: воспроизведение прежних отправок. Основной submission.csv не меняется.
if REPRODUCE_PREVIOUS:
    for variant, filename in [
        ("v4", "submission_v4_hgb_pool.csv"),
        ("v5", "submission_v5_hgb_v3_llrraw7.csv"),
        ("v7", "submission_v7_s10.csv"),
    ]:
        xt = builder.matrix(train_ids, train_ids, labels, variant=variant)
        xv = builder.matrix(test_ids, train_ids, labels, TEST_DAY, variant=variant)
        if variant == "v7":
            predictions = []
            for seed in range(42, 52):
                fitted = make_model(seed=seed, max_features=0.8).fit(xt, labels.loc[xt.index])
                predictions.append(fitted.predict_proba(xv)[:, 1])
            previous_scores = np.stack(predictions).mean(axis=0)
        else:
            fitted = make_model().fit(xt, labels.loc[xt.index])
            previous_scores = fitted.predict_proba(xv)[:, 1]
        write_submission(previous_scores, Path("previous") / filename, variant)
